<a href="https://colab.research.google.com/github/TamirPalay/DI_Exercises/blob/main/week13/day1-2/LoRA_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%pip install -U peft
!pip install datasets
%pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## Step 2: Loading pretrained model and tokenizer

In [8]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# bloomz doesn't have a pad token by default — needed for the data collator later
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

## Step 3: Load and preprocess dataset

In [9]:
dataset = load_dataset("Abirate/english_quotes")

# Shuffle for a random sample, then take 10% of the train split
full_train = dataset["train"].shuffle(seed=42)
data = full_train.select(range(int(len(full_train) * 0.1)))

# Tokenize the "quote" field
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

train_sample = data.select(range(5))
display(train_sample)

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})

In [10]:
print(train_sample[0])

{'quote': "“I don't mind making jokes, but I don't want to look like one.”", 'author': 'Marilyn Monroe', 'tags': ['appearance', 'jokes', 'marilyn-monroe'], 'input_ids': [123916, 5926, 19142, 16997, 21445, 262, 15, 1965, 473, 5926, 4026, 427, 5382, 3269, 2592, 17, 982], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## LoRA configuration

In [11]:
import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,                          # rank of the low-rank matrices — smaller = fewer trainable params
    lora_alpha=1,                 # scaling factor for the LoRA update
    target_modules=["query_key_value"],  # bloom's attention module name — this is where LoRA gets injected
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [12]:
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176
None


## Training arguments

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache/working", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,       # higher LR than full fine-tuning, since only a tiny fraction of params are updated
    num_train_epochs=5,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

## Save the model

In [ ]:
import time

time_now = time.strftime("%Y-%m-%d-%H-%M-%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path) #This only saves the small LoRA adapter weights, not the full base model — another PEFT space-saving win.

## Reload the model

In [ ]:
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False   # freeze it — we're just doing inference now
)

## Generate Text

In [ ]:
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))